In [2]:
%pip install pandas numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import pandas as pd
import numpy as np
df=pd.read_csv('D:\\DA\\Huawei_Health_Analysis\\data\\processed\\health_daily_summary_cleaned.csv')

#检查缺失值情况
df.isnull().sum()

日期        0
REM时长     0
午睡时长      0
浅睡时长      0
深睡时长      0
清醒时长      0
夜间总睡眠     0
在床时间      0
睡眠效率      0
深睡比例      0
浅睡比例      0
快速眼动比例    0
是否午睡      0
星期        0
总步数       0
总时长       0
基础代谢      0
活动消耗      0
全天总消耗     0
平均心率      0
最小心率      0
最大心率      0
记录条数      0
dtype: int64

In [4]:
#合格睡眠率
normal=len(df[(df['夜间总睡眠']>7)&(df['夜间总睡眠']<9)&(df['深睡比例']>20)&(df['浅睡比例']<60)])
total=len(df)
normal_sleep_rate=round(normal/total,2)
print("合格睡眠率为：",normal_sleep_rate)

conscious=round(df['清醒时长'].mean()/(df['夜间总睡眠'].mean()*60),2)
print(f"平均清醒率为：{conscious}")

#基于我的个人数据，清醒占比正常但合格的睡眠占比偏低，可能和晚睡有关。

合格睡眠率为： 0.4
平均清醒率为：0.06


In [29]:
#午睡对夜间睡眠效率的影响
平均睡眠效率=df['睡眠效率'].mean()
df['当晚睡眠效率']=df['睡眠效率'].shift(-1)
df['睡眠效率相对偏离']=round((df['当晚睡眠效率']-平均睡眠效率)/平均睡眠效率,2)
df[['日期','午睡时长','当晚睡眠效率','睡眠效率相对偏离']].sort_values(by='午睡时长',ascending=False).head(20)

#结论：基于我的个人数据，午睡过长几乎不会影响当晚的睡眠效率

,日期,午睡时长,当晚睡眠效率,睡眠效率相对偏离
115,2025-09-10,238.0,96.66,0.03
250,2026-01-23,215.0,95.91,0.02
126,2025-09-21,193.0,97.80,0.04
48,2025-07-05,173.0,97.68,0.04
207,2025-12-11,166.0,93.64,-0.01
167,2025-11-01,166.0,92.84,-0.01
90,2025-08-16,141.0,96.22,0.02
279,2026-02-21,132.0,94.77,0.01
108,2025-09-03,132.0,94.94,0.01
119,2025-09-14,128.0,96.13,0.02


In [ ]:
#按星期分组，观察不同星期对深睡比例的影响
round(df.groupby('星期').agg({'夜间总睡眠':'mean','深睡比例':'mean'}).sort_values(by='深睡比例',ascending=False).rename(columns={'夜间总睡眠':'均睡时长','深睡比例':'均深睡比例'}),2)

#结论：基于个人数据，一周内各天的深睡比例差异不大，可能和生活作息规律有关。

,均睡时长,均深睡比例
星期,,
周五,7.01,26.23
周日,7.43,25.84
周六,7.03,25.31
周一,7.86,25.07
周二,7.15,24.77
周四,6.99,24.76
周三,7.45,24.69


In [ ]:
#探讨第二天是否午睡和昨天睡眠时长的关系
df.groupby('是否午睡')['夜间总睡眠'].mean()

#结论：基于个人数据，昨天的睡眠时长平均减少半小时，第二天午睡的可能性增加，可能和睡眠不足有关。


是否午睡
False    7.639859
True     7.171208
Name: 夜间总睡眠, dtype: float64

In [ ]:
#探讨活动消耗与深睡比例、深睡时长的关系
df['当晚睡眠效率']=df['睡眠效率'].shift(-1)
df['当晚深睡比例']=df['深睡比例'].shift(-1)
df['当晚清醒时长']=df['清醒时长'].shift(-1)
df['消耗分段']=pd.cut(df['活动消耗'],bins=[0,500,1000,1500,2000,3000,5000],labels=['0-500','500-1000','1000-1500','1500-2000','2000-3000','3000+'])
round(df.groupby('消耗分段',observed=True).agg({'当晚深睡比例':'mean','当晚睡眠效率':'mean','当晚清醒时长':'mean'}).rename(columns={'当晚深睡比例':'均深睡比例','当晚睡眠效率':'均睡眠效率','当晚清醒时长':'均清醒时长'}),2)


#结论：基于个人数据，活动消耗越大，对当晚的深睡比例和睡眠效率均有提升，且可以减少夜间清醒，说明运动可以显著改善睡眠质量。

,均深睡比例,均睡眠效率,均清醒时长
消耗分段,,,
0-500,25.13,93.89,28.10
500-1000,24.94,94.00,27.59
1000-1500,25.09,94.90,23.14
1500-2000,27.66,95.10,22.58
2000-3000,28.43,95.44,22.00
3000+,29.33,96.78,15.75


In [ ]:
#计算睡眠指标与运动指标的相关性矩阵
df['当晚夜间总睡眠']=df['夜间总睡眠'].shift(-1)
df['当晚深睡时长']=df['深睡时长'].shift(-1)
df['当晚睡眠效率']=df['睡眠效率'].shift(-1)
df['当晚深睡比例']=df['深睡比例'].shift(-1)
df['当晚清醒时长']=df['清醒时长'].shift(-1)
corr_metrics=['当晚夜间总睡眠','当晚深睡时长','当晚清醒时长','当晚睡眠效率','当晚深睡比例','总步数','总时长','活动消耗','全天总消耗','平均心率']
corr_matrix=round(df[corr_metrics].corr(),2)
corr_matrix.loc[['总步数','总时长','活动消耗','全天总消耗','平均心率'],['当晚夜间总睡眠','当晚深睡时长','当晚睡眠效率','当晚深睡比例','当晚清醒时长']]

#结论：基于个人数据，所有运动指标都和深睡比例呈正相关，和清醒时长呈负相关，说明运动有助于增加深睡、减少夜间清醒。

,当晚夜间总睡眠,当晚深睡时长,当晚睡眠效率,当晚深睡比例,当晚清醒时长
总步数,-0.13,-0.03,0.05,0.07,-0.09
总时长,-0.07,0.03,-0.02,0.11,-0.00
活动消耗,0.10,0.14,0.11,0.09,-0.09
全天总消耗,0.10,0.14,0.11,0.09,-0.09
平均心率,0.09,0.14,0.02,0.10,-0.01


In [31]:
df['当晚睡眠效率']=df['睡眠效率'].shift(-1)
df['当晚深睡比例']=df['深睡比例'].shift(-1)
df['当晚清醒时长']=df['清醒时长'].shift(-1)
df['当晚浅睡比例']=df['浅睡比例'].shift(-1)
df.to_csv('daily_with_next_night.csv',index=False)